# Joint vs Late Fusion: Final Comparison

Runs once both notebooks have produced their test-set predictions into
`Results/`:

- `gemma4_predictions.csv`: columns `id, label, prob_hateful, pred`
- `late_fusion_predictions.csv`: columns `id, label, prob_text, prob_image, prob_late`

In [ ]:
import csv
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

## Configuration

In [ ]:
RESULTS_DIR = "../Results"
JOINT_CSV   = "gemma4_predictions.csv"
LATE_CSV    = "late_fusion_predictions.csv"
PLOT_OUT    = "roc_comparison.png"
SEED        = 42

## Metrics

AUROC, ROC curves, bootstrap CIs, and the fast DeLong paired test.

In [ ]:
def load_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return {row["id"]: row for row in csv.DictReader(f)}

def auroc(y, s):
    r = stats.rankdata(s)
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    return (r[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

def roc_curve(y, s):
    order = np.argsort(-s)
    y = y[order]
    tp, fp = np.cumsum(y), np.cumsum(1 - y)
    tpr = np.concatenate([[0], tp / tp[-1]])
    fpr = np.concatenate([[0], fp / fp[-1]])
    return fpr, tpr

def bootstrap_ci(y, s, n=5000, seed=SEED):
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    vals = []
    for _ in range(n):
        idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        vals.append(auroc(y[idx], s[idx]))
    return np.percentile(vals, [2.5, 97.5])

def _midrank(x):
    order = np.argsort(x)
    z = x[order]
    n = len(x)
    t = np.zeros(n)
    i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        t[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n)
    out[order] = t
    return out

def delong_paired(y, sa, sb):
    order = np.argsort(-y, kind="mergesort")
    y = y[order]
    m, n = int(y.sum()), len(y) - int(y.sum())
    preds = np.vstack([sa[order], sb[order]])
    tx = np.array([_midrank(preds[k, :m]) for k in range(2)])
    ty = np.array([_midrank(preds[k, m:]) for k in range(2)])
    tz = np.array([_midrank(preds[k]) for k in range(2)])
    aucs = tz[:, :m].sum(axis=1) / (m * n) - (m + 1) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    diff = aucs[0] - aucs[1]
    se = np.sqrt(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1])
    z = diff / se
    return {"diff": diff, "se": se, "z": z, "p": 2 * stats.norm.sf(abs(z))}

def bootstrap_diff(y, sa, sb, n=10000, seed=SEED):
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    diffs = np.empty(n)
    for b in range(n):
        idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        diffs[b] = auroc(y[idx], sa[idx]) - auroc(y[idx], sb[idx])
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"lo": lo, "hi": hi, "p_gt": float((diffs > 0).mean())}

## Load predictions

Join the two files on `id` and check the labels agree (guards against comparing mismatched runs).

In [ ]:
joint = load_csv(os.path.join(RESULTS_DIR, JOINT_CSV))
late  = load_csv(os.path.join(RESULTS_DIR, LATE_CSV))

ids = sorted(set(joint) & set(late))
assert ids, "no shared ids between the two prediction files"
for i in ids:
    assert int(joint[i]["label"]) == int(late[i]["label"]), f"label mismatch at id {i}"

y = np.array([int(joint[i]["label"]) for i in ids])
models = {
    "Gemma LoRA (joint)": np.array([float(joint[i]["prob_hateful"]) for i in ids]),
    "BERT text-only":     np.array([float(late[i]["prob_text"]) for i in ids]),
    "ViT image-only":     np.array([float(late[i]["prob_image"]) for i in ids]),
    "Late fusion (avg)":  np.array([float(late[i]["prob_late"]) for i in ids]),
}
print(f"n = {len(ids)}  |  {y.sum()} hateful / {(1 - y).sum()} not")

## AUROC per model

In [ ]:
rows = []
for name, s in models.items():
    lo, hi = bootstrap_ci(y, s)
    rows.append({"model": name, "AUROC": round(auroc(y, s), 4),
                 "AUROC 95% CI": f"[{lo:.3f}, {hi:.3f}]", "n": len(ids)})
pd.DataFrame(rows)

## Paired DeLong test: joint vs late

The headline result. Both models scored the same memes, so the test is paired.

In [ ]:
d = delong_paired(y, models["Gemma LoRA (joint)"], models["Late fusion (avg)"])
b = bootstrap_diff(y, models["Gemma LoRA (joint)"], models["Late fusion (avg)"])
verdict = "SIGNIFICANT" if d["p"] < 0.05 else "NOT significant"

print(f"diff (joint - late) = {d['diff']:+.3f}")
print(f"DeLong: SE = {d['se']:.3f}   z = {d['z']:.2f}   p = {d['p']:.3f}  ->  {verdict} at alpha=0.05")
print(f"bootstrap diff 95% CI [{b['lo']:+.3f}, {b['hi']:+.3f}]   P(joint > late) = {b['p_gt']:.3f}")

## Combined ROC

In [ ]:
plt.figure(figsize=(5.5, 5.5))
for name, s in models.items():
    fpr, tpr = roc_curve(y, s)
    plt.plot(fpr, tpr, label=f"{name} (AUROC = {auroc(y, s):.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"Joint vs Late Fusion (n = {len(ids)})")
plt.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, PLOT_OUT), dpi=150)
plt.show()